In [39]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split

from sklearn.preprocessing import MinMaxScaler, OneHotEncoder

from sklearn.linear_model import LogisticRegression
from sklearn.linear_model import Lasso, Ridge
from sklearn.feature_selection import RFE

## Import Dataset

In [2]:
cars = pd.read_csv('project_data/train.csv')
cars.head()

,carID,Brand,model,year,price,transmission,mileage,fuelType,tax,mpg,engineSize,paintQuality%,previousOwners,hasDamage
0,69512,VW,Golf,2016.0,22290,Semi-Auto,28421.0,Petrol,NaN,11.417268,2.0,63.0,4.000000,0.0
1,53000,Toyota,Yaris,2019.0,13790,Manual,4589.0,Petrol,145.0,47.900000,1.5,50.0,1.000000,0.0
2,6366,Audi,Q2,2019.0,24990,Semi-Auto,3624.0,Petrol,145.0,40.900000,1.5,56.0,4.000000,0.0
3,29021,Ford,FIESTA,2018.0,12500,anual,9102.0,Petrol,145.0,65.700000,1.0,50.0,-2.340306,0.0
4,10062,BMW,2 Series,2019.0,22995,Manual,1000.0,Petrol,145.0,42.800000,1.5,97.0,3.000000,0.0


In [47]:
cat_columns = cars.select_dtypes(exclude=np.number).columns.to_list()

## Drop Columns

In [42]:
cars.drop(columns=['hasDamage'], inplace=True)

KeyError: "['hasDamage'] not found in axis"

## Inconsistent Data

In [3]:
negative_features = ['mileage', 'mpg', 'engineSize', 'previousOwners', 'tax']

In [4]:
cars[negative_features] = cars[negative_features].abs()

In [5]:
cars[negative_features].min()

mileage           1.0
mpg               1.1
engineSize        0.0
previousOwners    0.0
tax               0.0
dtype: float64

## Training and Test

In [6]:
X = cars.drop('price', axis=1)
y = cars['price']

In [7]:
X_train, X_val, y_train, y_val = train_test_split(X,y, test_size=0.3, random_state=0, shuffle=True)

In [8]:
X_train.info()

<class 'pandas.core.frame.DataFrame'>
Index: 53181 entries, 74681 to 68268
Data columns (total 13 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   carID           53181 non-null  int64  
 1   Brand           52128 non-null  object 
 2   model           52136 non-null  object 
 3   year            52130 non-null  float64
 4   transmission    52086 non-null  object 
 5   mileage         52157 non-null  float64
 6   fuelType        52160 non-null  object 
 7   tax             47671 non-null  float64
 8   mpg             47683 non-null  float64
 9   engineSize      52118 non-null  float64
 10  paintQuality%   52091 non-null  float64
 11  previousOwners  52090 non-null  float64
 12  hasDamage       52087 non-null  float64
dtypes: float64(8), int64(1), object(4)
memory usage: 5.7+ MB


## Dealing with missing values

In [9]:
for feature in X_train.columns:
    if pd.api.types.is_numeric_dtype(X_train[feature]):
        
        fill_with_median = X_train[feature].median()

        X_train[feature].fillna(fill_with_median, inplace=True)
        X_val[feature].fillna(fill_with_median, inplace=True)

C:\Users\rafad\AppData\Local\Temp\ipykernel_11468\3777037498.py:6: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  X_train[feature].fillna(fill_with_median, inplace=True)
C:\Users\rafad\AppData\Local\Temp\ipykernel_11468\3777037498.py:7: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.



In [10]:
for column in ['Brand', 'model', 'transmission', 'fuelType']:
    X_train[column] = X_train[column].fillna('Unknown')
    X_val[column] = X_val[column].fillna('Unknown')

In [11]:
X_train.isna().sum()

carID             0
Brand             0
model             0
year              0
transmission      0
mileage           0
fuelType          0
tax               0
mpg               0
engineSize        0
paintQuality%     0
previousOwners    0
hasDamage         0
dtype: int64

## Data Types

In [17]:
numeric_cols = ['year', 'paintQuality%', 'previousOwners', 'hasDamage']

for feature in numeric_cols:
    X_train[feature] = X_train[feature].astype(int)

In [18]:
X_train.info()

<class 'pandas.core.frame.DataFrame'>
Index: 53181 entries, 74681 to 68268
Data columns (total 13 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   carID           53181 non-null  int64  
 1   Brand           53181 non-null  object 
 2   model           53181 non-null  object 
 3   year            53181 non-null  int64  
 4   transmission    53181 non-null  object 
 5   mileage         53181 non-null  float64
 6   fuelType        53181 non-null  object 
 7   tax             53181 non-null  float64
 8   mpg             53181 non-null  float64
 9   engineSize      53181 non-null  float64
 10  paintQuality%   53181 non-null  int64  
 11  previousOwners  53181 non-null  int64  
 12  hasDamage       53181 non-null  int64  
dtypes: float64(4), int64(5), object(4)
memory usage: 5.7+ MB


## Separate the categorical and numerical data

### Categorical

In [19]:
X_train_categorical = X_train.select_dtypes(exclude=np.number)

In [20]:
X_val_categorical = X_val.select_dtypes(exclude=np.number)

### Numerical

In [21]:
X_train_numerical = X_train.select_dtypes(include=np.number)

In [22]:
X_val_numerical = X_val.select_dtypes(include=np.number)

## Normalizing the data

In [23]:
scaler = MinMaxScaler()

scaler.fit(X_train_numerical)

X_train_numerical_scaled = scaler.transform(X_train_numerical)

In [24]:
X_train_numerical_scaled = pd.DataFrame(X_train_numerical_scaled, columns=X_train_numerical.columns).set_index(X_train.index)
X_train_numerical_scaled

,carID,year,mileage,tax,mpg,engineSize,paintQuality%,previousOwners,hasDamage
74681,0.559695,0.888889,0.004040,0.250000,0.086012,0.454545,0.435484,0.500000,0.0
47438,0.330591,0.833333,0.024152,0.250000,0.113264,0.242424,0.516129,0.333333,0.0
74144,0.593827,0.870370,0.057257,0.250000,0.113264,0.318182,0.766129,0.666667,0.0
53441,0.455746,0.888889,0.020690,0.258621,0.104961,0.212121,0.403226,0.500000,0.0
53978,0.874148,0.907407,0.039059,0.250000,0.077922,0.303030,0.427419,0.500000,0.0
...,...,...,...,...,...,...,...,...,...
21243,0.621193,0.907407,0.014848,0.250000,0.131360,0.242424,0.758065,0.166667,0.0
45891,0.945281,0.777778,0.303403,0.344828,0.099212,0.303030,0.790323,0.500000,0.0
42613,0.064631,0.907407,0.005570,0.250000,0.099638,0.151515,0.733871,0.333333,0.0
43567,0.884494,0.870370,0.092341,0.215517,0.120290,0.303030,0.322581,0.333333,0.0


In [25]:
X_val_numerical_scaled = scaler.transform(X_val_numerical)
X_val_numerical_scaled = pd.DataFrame(X_val_numerical_scaled, columns = X_val_numerical.columns).set_index(X_val.index)
X_val_numerical_scaled

,carID,year,mileage,tax,mpg,engineSize,paintQuality%,previousOwners,hasDamage
43620,0.487363,0.851852,0.157134,0.051724,0.134341,0.242424,0.516129,0.000000,0.0
3237,0.856786,0.833333,0.188852,0.275862,0.113264,0.212121,0.322581,0.166667,0.0
42099,0.384323,0.888889,0.027362,0.250000,0.120502,0.151515,0.693548,0.000000,0.0
56591,0.290904,0.907407,0.019254,0.250000,0.123057,0.151515,0.645161,1.043038,0.0
41334,0.926905,0.907407,0.019220,0.250000,0.099638,0.151515,0.258065,0.500000,0.0
...,...,...,...,...,...,...,...,...,...
71620,0.708464,0.888889,0.049403,0.250000,0.123057,0.227273,0.725806,0.500000,0.0
1877,0.650086,0.907407,0.006418,0.250000,0.090270,0.227273,0.298387,0.333333,0.0
34761,0.618955,0.870370,0.031966,0.250000,0.144347,0.242424,0.669355,0.166667,0.0
33158,0.353271,0.759259,0.394871,0.250000,0.113264,0.242424,0.411290,0.333333,0.0


## Feature Selection

#### Numerical

In [26]:
lasso_model = Lasso(alpha=0.1, random_state=42, max_iter=10000)

In [27]:
rfe_lasso = RFE(estimator=lasso_model, n_features_to_select=5)
rfe_lasso.fit(X_train_numerical_scaled, y_train)

RFE(estimator=Lasso(alpha=0.1, max_iter=10000, random_state=42),
    n_features_to_select=5)

In [28]:
rfe_lasso.support_

array([ True,  True,  True, False,  True,  True, False, False, False])

In [29]:
rfe_lasso.ranking_

array([1, 1, 1, 2, 1, 1, 3, 4, 5])

In [30]:
lasso_features_to_select = pd.Series(rfe_lasso.support_, index= X_train_numerical_scaled.columns)
lasso_features_to_select

carID              True
year               True
mileage            True
tax               False
mpg                True
engineSize         True
paintQuality%     False
previousOwners    False
hasDamage         False
dtype: bool

In [31]:
ridge_model = Ridge(alpha=0.1, random_state=42, max_iter=10000)

In [32]:
rfe_ridge = RFE(estimator=ridge_model, n_features_to_select=5)

In [33]:
rfe_ridge.fit(X_train_numerical_scaled, y_train)

RFE(estimator=Ridge(alpha=0.1, max_iter=10000, random_state=42),
    n_features_to_select=5)

In [34]:
rfe_ridge.support_

array([ True,  True,  True, False,  True,  True, False, False, False])

In [35]:
rfe_ridge.ranking_

array([1, 1, 1, 2, 1, 1, 3, 4, 5])

In [36]:
ridge_features_to_select = pd.Series(rfe_ridge.support_, index=X_train_numerical_scaled.columns)
ridge_features_to_select

carID              True
year               True
mileage            True
tax               False
mpg                True
engineSize         True
paintQuality%     False
previousOwners    False
hasDamage         False
dtype: bool

#### Categorical

‘ignore’ : When an unknown category is encountered during transform, the resulting one-hot encoded columns for this feature will be all zeros. In the inverse transform, an unknown category will be denoted as None.

In [50]:
ohe = OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore')
X_train_cat_encoded = pd.DataFrame(
    ohe.fit_transform(X_train_categorical),
    columns=ohe.get_feature_names_out(cat_columns),
    index=X_train_categorical.index
)

X_val_cat_encoded = pd.DataFrame(
    ohe.transform(X_val_categorical),
    columns=ohe.get_feature_names_out(cat_columns),
    index=X_val_categorical.index
)

c:\Users\rafad\anaconda3\Lib\site-packages\sklearn\preprocessing\_encoders.py:246: UserWarning: Found unknown categories in columns [0, 1] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


## Export

In [51]:
X_train_final = pd.concat([X_train_numerical_scaled, X_train_cat_encoded], axis=1)
X_val_final = pd.concat([X_val_numerical_scaled, X_val_cat_encoded], axis=1)

In [53]:
X_train_final

,carID,year,mileage,tax,mpg,engineSize,paintQuality%,previousOwners,hasDamage,Brand_AUDI,...,fuelType_etrol,fuelType_hybrid,fuelType_iese,fuelType_iesel,fuelType_other,fuelType_petro,fuelType_petrol,fuelType_ther,fuelType_ybri,fuelType_ybrid
74681,0.559695,0.888889,0.004040,0.250000,0.086012,0.454545,0.435484,0.500000,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
47438,0.330591,0.833333,0.024152,0.250000,0.113264,0.242424,0.516129,0.333333,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
74144,0.593827,0.870370,0.057257,0.250000,0.113264,0.318182,0.766129,0.666667,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
53441,0.455746,0.888889,0.020690,0.258621,0.104961,0.212121,0.403226,0.500000,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
53978,0.874148,0.907407,0.039059,0.250000,0.077922,0.303030,0.427419,0.500000,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
21243,0.621193,0.907407,0.014848,0.250000,0.131360,0.242424,0.758065,0.166667,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
45891,0.945281,0.777778,0.303403,0.344828,0.099212,0.303030,0.790323,0.500000,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
42613,0.064631,0.907407,0.005570,0.250000,0.099638,0.151515,0.733871,0.333333,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
43567,0.884494,0.870370,0.092341,0.215517,0.120290,0.303030,0.322581,0.333333,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [54]:
X_train_final.to_csv('X_train_preprocessed.csv', index=False)
X_val_final.to_csv('X_valPreprocessed.csv', index=False)